# Lipsync — run it without a computer

Infer what someone said from silent video of them speaking. Everything runs on
Google's machines, so a phone or tablet is enough.

## How to use this

1. Tap **Runtime → Run all** (on mobile: the **⋮** menu, then *Run all*).
2. Say yes to the "not authored by Google" warning.
3. Wait a few minutes for the first run — it installs things and downloads a 1 GB model.
4. The **last cell prints a public link**. Open it. That is the app.

The link works in any browser on any device, and keeps working while this
notebook stays open. If the notebook disconnects (Colab does this after about 90
minutes of inactivity), run it again to get a fresh link.

## Before you believe anything it tells you

**This produces guesses that read as certainties.** The model gets roughly one
word in five wrong on clean, head-on, well-lit video, and considerably worse on
anything else. Many sounds are visually identical — `p`, `b` and `m` are the same
picture, as are `f` and `v` — so a language model fills the gaps, and it returns
fluent English whether or not it actually read anything.

Never use the output to claim a particular person said a particular thing.

## Step 1 — install

Takes two or three minutes the first time.

In [ ]:
REPO = "https://github.com/TridentIntelFree/Lipsync-.git"

# The code currently lives on a feature branch. If that branch is ever merged
# and deleted, fall back to main rather than breaking this notebook.
BRANCHES = ["claude/lipsync-speech-inference-k649jo", "main"]

import os, subprocess, sys

if not os.path.isdir("/content/Lipsync-"):
    for branch in BRANCHES:
        done = subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", branch, REPO, "/content/Lipsync-"],
            capture_output=True, text=True,
        )
        if done.returncode == 0:
            print(f"Cloned from branch: {branch}")
            break
    else:
        raise SystemExit(f"Could not clone {REPO} from any of {BRANCHES}")

os.chdir("/content/Lipsync-")

# torch ships with Colab already; everything else is small. gradio is pinned to
# 6+ because app.py uses its current launch() API. No espnet and no torchaudio:
# the visual model build is vendored in the repo. See VENDOR.md.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "mediapipe", "opencv-python-headless", "imageio-ffmpeg", "gradio>=6"],
    check=True,
)
sys.path.insert(0, "/content/Lipsync-")

import torch
device = "GPU" if torch.cuda.is_available() else "CPU (slow — Runtime > Change runtime type > T4 GPU)"
print(f"Installed. Running on: {device}")

# Fail here, loudly, rather than three cells later with a confusing error.
from lipsync.backend import _use_vendored_espnet
_use_vendored_espnet()
from espnet.nets.pytorch_backend.e2e_asr_transformer import E2E  # noqa: F401
print("Recognition backend imports cleanly.")

## Step 2 — download the model

About 1 GB, from public HuggingFace links. No account or token needed. This is
cached, so re-running the notebook later in the same session is instant.

In [ ]:
from lipsync.recognize import download_weights, weights_present

if weights_present():
    print("Weights already downloaded.")
else:
    for name, path in download_weights().items():
        print(f"  {name}: {path}")

print("\nReady.")

## Step 3 — start the app

This prints two links. Use the **public URL** (it ends in `.gradio.live`) — that
is the one that works from your phone.

Leave this cell running while you use the app. Stopping it kills the link.

In [ ]:
import app

app.build().launch(share=True, theme=__import__("gradio").themes.Soft())

## Getting a usable result

Face the camera straight on, fill a good part of the frame with your head, use
even front lighting, and keep your mouth unobstructed. A selfie video at arm's
length is close to ideal. Wide shots and side angles will not work.

The app shows a filmstrip of the aligned mouth crops it actually fed the model.
**If that strip is not centred on a mouth, ignore the transcript** — no matter
how convincing the sentence looks.

Untick *Attempt a transcript* to check whether footage is usable without waiting
for recognition. That path needs no model and is quick.